# 03 — Linear Models

Regression and classification via linear combinations of features, with regularization variants.

## Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)
lr.predict(X_test)
lr.coef_, lr.intercept_
lr.score(X_test, y_test)   # R^2


## Regularized Regression: Ridge, Lasso, ElasticNet

In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet, RidgeCV, LassoCV

ridge = Ridge(alpha=1.0)          # L2 penalty — shrinks coefficients, keeps all features
lasso = Lasso(alpha=0.1)          # L1 penalty — can zero out coefficients (feature selection)
elastic = ElasticNet(alpha=0.1, l1_ratio=0.5)   # mix of L1 and L2

ridge.fit(X_train, y_train)
lasso.fit(X_train, y_train)
(lasso.coef_ == 0).sum()   # count of features zeroed out by L1

# built-in CV versions pick alpha automatically
ridge_cv = RidgeCV(alphas=[0.1, 1.0, 10.0], cv=5).fit(X_train, y_train)
ridge_cv.alpha_


## Logistic Regression (classification, despite the name)

In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(penalty="l2", C=1.0, max_iter=1000)   # C = inverse regularization strength
clf.fit(X_train, y_train)
clf.predict(X_test)
clf.predict_proba(X_test)     # class probabilities
clf.decision_function(X_test) # raw scores before thresholding

# multiclass: handled automatically (multinomial by default for lbfgs/saga solvers)
clf_multi = LogisticRegression(multi_class="multinomial")


## SGD-based Linear Models (for large datasets)

In [ ]:
from sklearn.linear_model import SGDClassifier, SGDRegressor

sgd_clf = SGDClassifier(loss="log_loss", penalty="l2", max_iter=1000)  # loss="hinge" gives linear SVM
sgd_clf.fit(X_train, y_train)

sgd_reg = SGDRegressor(loss="squared_error", penalty="l2")
sgd_reg.fit(X_train, y_train)


## Polynomial Regression (linear model + engineered features)

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

poly_model = make_pipeline(
    PolynomialFeatures(degree=3),
    StandardScaler(),
    Ridge(alpha=1.0),
)
poly_model.fit(X_train, y_train)


## Regularization Path Visualization

In [ ]:
import matplotlib.pyplot as plt

alphas = np.logspace(-3, 3, 50)
coefs = []
for a in alphas:
    ridge = Ridge(alpha=a).fit(X_train, y_train)
    coefs.append(ridge.coef_)

plt.plot(alphas, coefs)
plt.xscale("log")
plt.xlabel("alpha")
plt.ylabel("coefficients")
plt.title("Ridge regularization path")
plt.show()


## Common Gotchas
- Regularized models (Ridge/Lasso/LogisticRegression) are sensitive to feature scale — always scale features first.
- Lasso can behave unstably with correlated features (arbitrarily picks one); ElasticNet is more stable in that case.
- `LogisticRegression`'s `C` is inverse of regularization strength (smaller `C` = stronger regularization) — opposite convention from `alpha` in Ridge/Lasso.
- Increase `max_iter` if you see `ConvergenceWarning` — the default (100) is often too low for unscaled or high-dimensional data.